In [ ]:
from ecmwf.opendata import Client
import xarray as xr, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cf

Client(source="ecmwf").retrieve(step=24, type="fc", param=["2t", "msl"],
                                target="tahmin.grib2")      # 24 saat sonrası

def oku(ad):
    ds = xr.open_dataset("tahmin.grib2", engine="cfgrib",
                         filter_by_keys={"shortName": ad}, indexpath="")
    return ds[list(ds.data_vars)[0]].sel(latitude=slice(44, 34),
                                         longitude=slice(24, 47))

T = oku("2t") - 273.15     # Kelvin → °C
P = oku("msl") / 100       # Pa → hPa

ax = plt.figure(figsize=(11, 5)).add_subplot(projection=ccrs.PlateCarree())
cs = ax.contourf(T.longitude, T.latitude, T, levels=20, cmap="coolwarm")
iso = ax.contour(P.longitude, P.latitude, P, levels=range(980, 1044, 4),
                 colors="k", linewidths=0.8)
ax.clabel(iso, fmt="%d")
ax.coastlines(); ax.add_feature(cf.BORDERS, linestyle=":")
plt.colorbar(cs, label="°C")
plt.title(f"ECMWF tahmini – {str(T.valid_time.values)[:16]}"); plt.show()